# A reform arrives halfway through the story

**A guided investigation · Chapter 6 · 30–45 minutes · No Python experience needed**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cmg777/intro2causal/blob/main/notebooks_colab/explore-synthetic_did.ipynb)

> **The question**  
> How do schooling and earnings trajectories differ around a simulated reform?

A policy often arrives after years of change are already underway. This simulation follows 20 states from 1995 to 2014, with ten adopting a schooling reform in 2005. Each row is a state-year average. The generator gives states different starting levels, a shared underlying time trend, and random fluctuations; after adoption, it adds a schooling increase and a change in average log earnings to treated states. We will first inspect the trajectories, then compare schooling patterns and changes relative to a common pre-reform year. Looking at changes helps separate starting differences from subsequent movement. In this artificial world, we can inspect the assumptions written into the generator. In an actual policy study, we would have to argue for those assumptions using evidence and institutional knowledge rather than knowing them by construction.

### By the end, you will be able to…

- Compare levels and changes, then explain the counterfactual role of the comparison states.
- Read and modify three interactive charts using short Python examples.
- Explain one reason a descriptive pattern alone cannot settle this case’s causal question.

### Before you see the evidence

If treated states start with lower earnings, does that alone make them an unsuitable comparison?

Write two sentences now. You will return to them at the end.

**Your route:** meet the data → inspect three views → modify and try → ask the causal question.

## Your first few minutes in Colab · 3 minutes

You do not need prior Python experience. A **cell** is one block of text or code. Read the text, then press the **▶** beside the next code cell (or Shift+Enter). Its output appears underneath. Run cells in order: later cells use tables created earlier.

Choose **File → Save a copy in Drive** if you want to keep your changes. If a name is “not defined,” run the earlier setup cells again. **Runtime → Restart session and run all** gives you a fresh start. An error does not mean you have damaged the data: this notebook only reads it.

Keep a short learning log in a text cell. Before each chart, make a prediction. Afterward, write one observation with a number and one question that the chart cannot answer.

**Chart controls:** hover for values, click legend labels to hide/show a group, drag to zoom, and double-click to reset. A chart’s menu also offers a reset/autoscale control.

The setup installs Plotly Express if needed. `import` makes a library available; `pd` and `px` are short names for pandas (tables) and Plotly Express (charts).

In [ ]:
%pip install -q "plotly>=6,<7"

In [ ]:
import pandas as pd
import plotly.express as px
from IPython.display import display

# One consistent chart style; no plotting experience needed.
px.defaults.template = "plotly_white"
px.defaults.color_continuous_scale = "Blues"
px.defaults.color_discrete_sequence = ["#2563eb", "#7c3aed", "#059669", "#64748b"]

## Meet the data · 5 minutes

**One row:** Simulated state-year.  
**Stored size:** 400 rows × 6 columns.

| Column | What it records |
|---|---|
| `year` | Calendar year of the observation; census year in the child-labor file. |
| `treated` | Indicator for simulated reform states 1–10 (1), versus comparison states 11–20 (0). |
| `post` | Indicator for simulated years 2005 onward. |
| `avg_schooling` | Simulated state-year average years of schooling. |
| `avg_earnings` | Simulated state-year average log earnings, not dollars. |

Rows are repeated state-years, not independent people. avg_earnings is an average log-earnings outcome. Similar pre-trends are evidence to consider, not proof of future parallel trends.

`read_csv` downloads a table. We call it `df`. `head(5)` shows just five rows, not a separate sample used for the analysis. Square brackets select columns by their names.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/cmg777/intro2causal/main/data/ch6/synthetic_did.csv"
df = pd.read_csv(DATA_URL)
print(f"Loaded {len(df):,} rows and {len(df.columns)} columns")
display(df[['year', 'treated', 'post', 'avg_schooling', 'avg_earnings']].head(5))

### Pause: what do the rows mean?

Choose one displayed row. Translate it into a sentence using the variable guide. Which values are categories or identifiers? Which are quantities? A numeric storage type does not necessarily make a code a meaningful measurement.

The next table checks types, missing values, and the number of distinct values for the key columns. `isna()` marks missing cells; `sum()` counts them. Missing is not zero. Ask which comparison would lose a row if one of its required values were missing.

In [ ]:
key_columns = ['year', 'treated', 'post', 'avg_schooling', 'avg_earnings']
checks = pd.DataFrame({"storage_type": df[key_columns].dtypes.astype(str),
                       "missing_rows": df[key_columns].isna().sum(),
                       "distinct_values": df[key_columns].nunique()})
display(checks)
print("The charts below omit missing values required for each plotted quantity.")

### A small summary before plotting

`describe()` summarizes numeric columns: `count` counts available values, `mean` is the average, and `50%` is the median. The median splits ordered values into two halves. Averages of identifiers or arbitrary category codes are not useful descriptions, even if Python calculates them.

For category-like key variables, the second table counts values, including missing entries. Check these labels before interpreting a chart.

In [ ]:
numeric_keys = df[key_columns].select_dtypes(include="number").columns
if len(numeric_keys):
    display(df[numeric_keys].describe().round(2))
for column in key_columns:
    if df[column].nunique() <= 8:
        display(df[column].value_counts(dropna=False).rename("Rows").to_frame())

## Read the chart code · 2 minutes

`px.bar`, `px.scatter`, `px.line`, and `px.box` choose a chart type. `x` and `y` name columns for the axes; `color` distinguishes groups or values. `labels` translates column names for readers. `fig.show()` displays the result.

A table transformation and a plot answer different questions: first decide what each plotted row represents, then choose how to display it. Before running each chart, predict what you expect to see.

In [ ]:
# Display labels: run once; you do not need to memorize this mapping.
labels = {'ability_group': 'Ability group',
 'avg_earnings': 'Mean log earnings',
 'avg_schooling': 'Mean years of schooling',
 'census': 'Census year',
 'group': 'State group',
 'offer': 'Scholarship assignment',
 'plan': 'Assigned insurance plan',
 'state': 'State code',
 'state_name': 'Simulated state',
 'year': 'Year'}

## Visual investigation 1 · Look at the full trajectory

**Predict:** Do the state groups start at the same level and move in the same direction?

We average equally across states within each year and treatment group. The vertical guide marks the reform year, 2005.

In [ ]:
trends = df.groupby(["year", "treated"], as_index=False)[["avg_schooling", "avg_earnings"]].mean()
trends["group"] = trends["treated"].map({0: "Comparison states", 1: "Reform states"})
fig = px.line(trends, x="year", y="avg_earnings", color="group", markers=True, labels=labels, title="Levels differ before the reform")
fig.add_vline(x=2005, line_dash="dash")
fig.show()

**Explore the chart:** Compare the pre-2005 slopes separately from the levels. Hover to inspect the reform year.

**Write before reading on:** Describe one pattern with its unit and comparison group. Then give one possible explanation.

<details>
<summary>Interpretation checkpoint</summary>

Parallel movement does not require equal levels. In this simulation the shared trend is imposed by the generator; real evidence cannot establish future parallel trends with certainty.

</details>

## Visual investigation 2 · Follow the schooling mechanism

**Predict:** Does the schooling outcome shift at the same time?

Reuse the same group-year table and change the outcome. This helps keep the comparison consistent across outcomes.

In [ ]:
fig = px.line(trends, x="year", y="avg_schooling", color="group", markers=True, labels=labels, title="The reform and years of schooling")
fig.add_vline(x=2005, line_dash="dash")
fig.show()

**Explore the chart:** Toggle the comparison group off and on. What would a before/after reading miss without it?

**Write before reading on:** Describe one pattern with its unit and comparison group. Then give one possible explanation.

<details>
<summary>Interpretation checkpoint</summary>

Schooling was rising before the reform in both groups. The comparison trajectory helps us ask how much change would have occurred anyway.

</details>

## Visual investigation 3 · Compare changes from a common reference

**Predict:** What happens when both paths start at zero in 2004?

Subtract each group’s 2004 log-earnings mean from its full trajectory. Zero now means no change from that group’s own baseline.

In [ ]:
base = trends[trends["year"] == 2004].set_index("group")["avg_earnings"]
trends["change"] = trends["avg_earnings"] - trends["group"].map(base)
fig = px.line(trends, x="year", y="change", color="group", markers=True, labels={"year": "Year", "change": "Change in mean log earnings since 2004", "group": "State group"}, title="A common baseline makes divergence visible")
fig.add_vline(x=2005, line_dash="dash")
fig.show()

**Explore the chart:** Compare the post-reform separation with pre-reform variation. Do not confuse log points with dollars.

**Write before reading on:** Describe one pattern with its unit and comparison group. Then give one possible explanation.

<details>
<summary>Interpretation checkpoint</summary>

The relative change is suggestive because the generator imposes the needed counterfactual trend. For real reforms, other group-specific shocks could produce the same picture.

</details>

## Make it yours · 8 minutes

### 1. Modify an example

Change the normalization year from 2004 to 2000 and explain what changes and what does not.

Copy that chart cell before editing, so you can compare the original and modified outputs. Say what changed in your question as well as in your code.

### 2. Try a small investigation

Plot individual schooling trajectories for reform states 1, 2, and 3.

Write a prediction first. The optional starter cell below is safe to leave unchanged; later cells do not depend on your answer.

In [ ]:
# Your investigation goes here.
# Describe your expected pattern before writing code.

<details>
<summary>Hint 1 · choose the right evidence</summary>

Filter state with isin([1, 2, 3]) and sort by year.

</details>

<details>
<summary>Hint 2 · a Python starting point</summary>

Convert state to a string label so Plotly draws a separate line for each state.

</details>

<details>
<summary>Optional worked solution · open after trying</summary>

<pre><code>states = df[df[&quot;state&quot;].isin([1, 2, 3])].sort_values(&quot;year&quot;).copy()
states[&quot;state_name&quot;] = states[&quot;state&quot;].astype(str)
fig = px.line(states, x=&quot;year&quot;, y=&quot;avg_schooling&quot;, color=&quot;state_name&quot;, markers=True, labels=labels)
fig.add_vline(x=2005, line_dash=&quot;dash&quot;)
fig.show()</code></pre>

Group averages hide state-specific noise. An individual state can wiggle around the common pattern without contradicting the simulation’s underlying trend.

Copy the code into the starter cell to run it.

</details>

## What can you now say? · 4 minutes

Return to your initial prediction. Write a four-sentence evidence note:

1. **I expected…** State your prediction.
2. **I observed…** Use a chart, a comparison, and the correct units.
3. **This does not establish…** Name a causal claim the evidence cannot settle.
4. **To learn more, I would need…** Propose an additional measurement or a more credible comparison.

### Carry this question into the chapter

**What would threaten the claim that comparison states describe how treated states would have changed without reform?**

[Continue with Chapter 6: The Wages of Schooling](https://cmg777.github.io/intro2causal/book/_book/notebooks_quarto/06-wages-of-schooling.html#strategy-5-differences-in-differences) · [Return to the case collection](https://cmg777.github.io/intro2causal/data-dictionary.html#ch6-synthetic-did-csv)

### Source and preparation notes

[Repository teaching simulation, code/python/ch6/generate_synthetic.py](https://github.com/cmg777/intro2causal/blob/main/code/python/ch6/generate_synthetic.py)

Generated by code/python/ch6/generate_synthetic.py after the OVB and RCT draws without resetting the seed. States 1–10 are treated; treatment starts in 2005. State baselines are drawn once per state, followed by schooling and earnings noise for each year.

These exploratory charts are based on the committed CSV. They are not a complete replication of the original study. Keep the source study, this prepared extract, and your descriptive results distinct.